# 7. The Lagrangian and the dual function  [CO 14.1–14.3]

TOML-MIRI — Maximum Likelihood Estimation and Convex Optimization, exploration exercises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed

np.set_printoptions(precision=4, suppress=True)
from scipy.optimize import minimize_scalar, minimize

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

---
## 7.1 Visualising the Lagrangian

**Task.** For the example of [CO 14.1.1],

$$\min_x\ \tfrac14x^2 - x\quad\text{s.t.}\quad x\le0, \qquad L(x,\lambda) = \tfrac14x^2 - x + \lambda x,$$

plot (i) the surface $L(x,\lambda)$ over $x\in[-4,4]$, $\lambda\in[0,3]$, and (ii) the curves $L(\cdot,\lambda)$ for $\lambda = 0, 0.5, 1, 2, 3$ together with $f_0(x) + I_{\mathbb R_-}(x)$.

### Prediction

- For fixed $x$, $L$ is linear in $\lambda$; for fixed $\lambda$ it is a parabola in $x$ with minimum at $x = 2(1-\lambda)$.
- On the feasible side $x\le0$ every curve is below $f_0$ (since $\lambda x\le0$); on the infeasible side they rise above $f_0$ but never reach the $+\infty$ wall of the indicator.

In [ ]:
f0 = lambda x: 0.25 * x**2 - x
f1 = lambda x: x
L = lambda x, lam: f0(x) + lam * f1(x)

xg, lg = np.linspace(-4, 4, 121), np.linspace(0, 3, 91)
XG, LG = np.meshgrid(xg, lg)
fig = plt.figure(figsize=(16, 5.5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot_surface(XG, LG, L(XG, LG), cmap="viridis", lw=0, alpha=0.9)
ax.set_xlabel("x"); ax.set_ylabel("λ"); ax.set_zlabel("L(x, λ)"); ax.set_title("L(x, λ) = ¼x² − x + λx")
ax2 = fig.add_subplot(1, 2, 2)
x = np.linspace(-4, 4, 400)
for lam in [0, 0.5, 1, 2, 3]:
    ax2.plot(x, L(x, lam), label=f"λ = {lam}")
ax2.plot(x[x <= 0], f0(x[x <= 0]), "k", lw=3, label="f₀ + I_{ℝ₋}  (= +∞ for x > 0)")
ax2.axvline(0, color="k", lw=3); ax2.set_ylim(-4, 9); ax2.legend(); ax2.grid(alpha=0.3); ax2.set_xlabel("x")
plt.show()

### Verify numerically

In [ ]:
xf = np.linspace(-10, 0, 2001); lams = np.linspace(0, 100, 1001)
XF, LF = np.meshgrid(xf, lams)
check("L(x, λ) ≤ f₀(x) for every feasible x ≤ 0 and every λ ≥ 0", np.all(L(XF, LF) <= f0(XF) + 1e-12))
check("λ f₁(x) ≤ I_{ℝ₋}(x) for all x and λ ≥ 0 (≤ 0 where x ≤ 0)", np.all(LF * XF <= 0))
xi = 0.5
check("for infeasible x = 0.5, sup over λ of λ f₁(x) grows without bound", (lams * xi).max() == 50)

### Explore: λ = 10, 100, and $\lambda f_1(x)$ next to the indicator

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))
for lam in [0, 1, 3, 10, 100]:
    ax1.plot(x, L(x, lam), label=f"λ = {lam}")
    ax2.plot(x, lam * f1(x), label=f"λ f₁(x), λ = {lam}")
ax1.plot(x[x <= 0], f0(x[x <= 0]), "k", lw=3); ax1.set_ylim(-20, 20); ax1.set_title("L(·, λ)")
ax2.plot(x[x <= 0], 0 * x[x <= 0], "k", lw=4, label="I_{ℝ₋}(x) (0 on x ≤ 0, +∞ on x > 0)"); ax2.axvline(0, color="k", lw=4)
ax2.set_ylim(-20, 20); ax2.set_title("the penalty λx versus the indicator")
for a in (ax1, ax2): a.legend(fontsize=8); a.grid(alpha=0.3)
plt.show()

---
### Explain

**1. Check that $L(x,\lambda)\le f_0(x)$ for every feasible $x$ and $\lambda\ge0$ [CO 14.1.4].**

For feasible $x$ ($x\le0$) and $\lambda\ge0$, $\lambda f_1(x) = \lambda x\le0$, so $L = f_0 + \lambda f_1\le f_0$. Checked on a grid $x\in[-10,0]$, $\lambda\in[0,100]$. In the plot every coloured curve lies below the black curve on the left half. This is the whole basis of weak duality: minimising $L$ over *all* $x$ gives something no larger than minimising $f_0$ over the feasible $x$.

**2. In what sense is $\lambda f_1(x)$ a (bad) approximation of the indicator function, and why is $\sup_{\lambda\ge0}$ of it exactly the indicator [CO 14.1.3]?**

The indicator $I_{\mathbb R_-}$ is 0 on the feasible set and $+\infty$ outside, a "hard wall". The penalty $\lambda x$ is a **linear under-estimator** of it: $\le 0$ (instead of $=0$) on the feasible side, and finite with slope $\lambda$ (instead of $+\infty$) on the infeasible side. It is a bad approximation for any single $\lambda$: it even rewards feasible points for being deep inside, and it only charges a linear price for violation. But taking the supremum over $\lambda\ge0$ fixes both defects: for $x\le0$ the best choice is $\lambda = 0$, giving 0; for $x>0$, $\lambda x\to+\infty$ as $\lambda\to\infty$. So $\sup_{\lambda\ge0}\lambda f_1(x) = I_{\mathbb R_-}(x)$ exactly, and the constrained problem is $\min_x\sup_{\lambda\ge0}L(x,\lambda)$. Duality swaps the order: $\max_\lambda\min_x L$.

---
## 7.2 Visualising the dual function

**Task.** For the same example, compute $g(\lambda) = \inf_x L(x,\lambda)$ numerically on a grid of $\lambda$ (minimise over $x$ for each $\lambda$) and analytically, and plot it together with the horizontal line $p^\star$. On the curves of the previous exercise, mark the minimum of each $L(\cdot,\lambda)$.

### Prediction

- $\partial_xL = \frac x2 - 1 + \lambda = 0\Rightarrow x(\lambda) = 2(1-\lambda)$ and $g(\lambda) = -(1-\lambda)^2$: a downward parabola touching $p^\star = 0$ at $\lambda = 1$.

In [ ]:
lams = np.linspace(0, 3, 61)
g_num = np.array([minimize_scalar(lambda x: L(x, l)).fun for l in lams])
g_formula = -(1 - lams) ** 2
p_star = 0.0          # x* = 0, f₀(0) = 0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))
for lam in [0, 0.5, 1, 2, 3]:
    ax1.plot(x, L(x, lam), label=f"λ = {lam}"); xm = 2 * (1 - lam)
    ax1.plot(xm, L(xm, lam), "ko")
ax1.plot(x[x <= 0], f0(x[x <= 0]), "k", lw=3); ax1.axvline(0, color="k", lw=3)
ax1.set_ylim(-5, 9); ax1.legend(); ax1.grid(alpha=0.3); ax1.set_title("L(·, λ) and its minimum (black dots)")
ax2.plot(lams, g_num, "o", ms=4, label="g(λ) numerical"); ax2.plot(lams, g_formula, "k", label="−(1 − λ)²")
ax2.axhline(p_star, color="r", ls="--", label="p* = 0"); ax2.plot(1, 0, "r*", ms=15, label="d* = g(1) = 0")
ax2.legend(); ax2.grid(alpha=0.3); ax2.set_xlabel("λ"); ax2.set_title("dual function")
plt.show()

### Verify numerically

In [ ]:
check("numerical inf_x L(x, λ) = −(1 − λ)²", np.allclose(g_num, g_formula, atol=1e-8))
check("g(λ) ≤ p* for every λ ≥ 0", np.all(g_num <= p_star + 1e-12))
check("g is concave (second differences ≤ 0)", np.all(np.diff(g_num, 2) <= 1e-9))
check("d* = max g = 0 at λ* = 1  ⇒  no duality gap", np.isclose(g_num.max(), p_star) and np.isclose(lams[g_num.argmax()], 1))

---
### Explain

**1. Show that $g(\lambda) = -(1-\lambda)^2$.**

$L(x,\lambda) = \frac14x^2 + (\lambda-1)x$ is a convex parabola in $x$. $\partial_xL = \frac x2 + \lambda - 1 = 0$ gives $x(\lambda) = 2(1-\lambda)$, and plugging in: $g(\lambda) = \frac14\cdot4(1-\lambda)^2 + (\lambda-1)\cdot2(1-\lambda) = (1-\lambda)^2 - 2(1-\lambda)^2 = -(1-\lambda)^2$. The numerical minimisation agrees to $10^{-8}$.

**2. Why is $g$ concave and below $p^\star$ for every $\lambda$ [CO 14.2.1–14.2.2]?**

**Concave:** for each fixed $x$, $L(x,\lambda) = f_0(x) + \lambda f_1(x)$ is an affine function of $\lambda$; $g$ is the pointwise infimum of this family of affine functions (the lower envelope of the lines in the notes' figure), and an infimum of affine functions is concave, whatever $f_0, f_1$ are. **Below $p^\star$:** $g(\lambda) = \inf_{x\in D}L\le\inf_{x\text{ feasible}}L\le\inf_{x\text{ feasible}}f_0 = p^\star$, using $L\le f_0$ on the feasible set (7.1).

**3. Where is $d^\star = \max_\lambda g(\lambda)$ attained, and is there a duality gap?**

At $\lambda^\star = 1$, with $d^\star = g(1) = 0 = p^\star$: **no duality gap** (strong duality). This is expected: the problem is convex and Slater holds ($x = -1$ is strictly feasible). At $\lambda^\star = 1$ the minimiser of $L(\cdot,1)$ is $x(1) = 0 = x^\star$, the primal solution: the dual optimum "points to" the primal optimum. Also note $\lambda^\star f_1(x^\star) = 1\cdot0 = 0$ (complementary slackness).

---
## 7.3 Duality in pictures: the set $\mathcal G$

**Task.** For a problem with one inequality constraint, plot the set $\mathcal G = \{(f_1(x), f_0(x)) : x\in\mathcal D\}$ in the $(u, t)$ plane. For several $\lambda\ge0$ draw the line $t + \lambda u = g(\lambda)$, the lowest line of slope $-\lambda$ touching $\mathcal G$. Its intercept with the $t$ axis is $g(\lambda)$. Do it for the example above and for a non-convex $f_0$ (e.g. $f_0(x) = x^4 - 3x^2 + x$, $f_1(x) = x - 0.5$).

In the picture: $p^\star = \min\{t : (u,t)\in\mathcal G,\ u\le0\}$ (lowest point of $\mathcal G$ on the left half-plane) and $d^\star = \max_\lambda g(\lambda)$ (highest intercept of a supporting line).

### Prediction

- Example of 7.1: $\mathcal G$ is the parabola $t = u^2/4 - u$; the supporting line with $\lambda = 1$ is horizontal and touches at $(0, 0)$: $p^\star = d^\star = 0$.
- Non-convex example: the curve has two "valleys". Whether there is a gap depends on whether the constraint cuts off the global minimum (see below).

In [ ]:
def g_of(f0, f1, lam, xs):
    return np.min(f0(xs) + lam * f1(xs))

def plot_G(ax, f0, f1, xs, lams, title, ulim=None, tlim=None):
    U, T = f1(xs), f0(xs)
    feas = U <= 0
    ax.plot(U, T, "k", lw=2, label="G = {(f₁(x), f₀(x))}")
    p_star = T[feas].min(); x_p = xs[feas][T[feas].argmin()]
    gs = np.array([g_of(f0, f1, l, xs) for l in lams])
    uu = np.linspace(*(ulim or (U.min(), U.max())), 2)
    for l, gv in zip(lams[::max(1, len(lams) // 6)], gs[::max(1, len(lams) // 6)]):
        ax.plot(uu, gv - l * uu, lw=1, alpha=0.8, label=f"λ = {l:.2f}: intercept g = {gv:.2f}")
    d_star = gs.max(); l_star = lams[gs.argmax()]
    ax.axvline(0, color="gray", lw=0.8); ax.axhline(p_star, color="r", ls="--", label=f"p* = {p_star:.3f}")
    ax.axhline(d_star, color="b", ls=":", label=f"d* = {d_star:.3f} (λ* = {l_star:.2f})")
    ax.plot(U[feas][T[feas].argmin()], p_star, "r*", ms=14)
    ax.set_xlabel("u = f₁(x)"); ax.set_ylabel("t = f₀(x)"); ax.legend(fontsize=7); ax.grid(alpha=0.3); ax.set_title(title)
    if ulim: ax.set_xlim(*ulim)
    if tlim: ax.set_ylim(*tlim)
    return p_star, d_star

xs = np.linspace(-6, 6, 200_001)
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
pd1 = plot_G(axes[0], f0, f1, xs, np.linspace(0, 3, 13), "example 7.1: f₀ = ¼x² − x, f₁ = x", ulim=(-4, 4), tlim=(-3, 8))
f0n = lambda x: x**4 - 3 * x**2 + x
pd2 = plot_G(axes[1], f0n, lambda x: x - 0.5, np.linspace(-2.2, 2.2, 200_001), np.linspace(0, 6, 25),
             "f₀ = x⁴ − 3x² + x, f₁ = x − 0.5", ulim=(-2.7, 1.7), tlim=(-5, 6))
pd3 = plot_G(axes[2], f0n, lambda x: 0.5 - x, np.linspace(-2.2, 2.2, 200_001), np.linspace(0, 6, 25),
             "same f₀, f₁ = 0.5 − x  (i.e. x ≥ 0.5)", ulim=(-1.7, 2.7), tlim=(-5, 6))
plt.show()
for name, (p, d) in [("7.1 example", pd1), ("non-convex, x ≤ 0.5", pd2), ("non-convex, x ≥ 0.5", pd3)]:
    print(f"{name:<22} p* = {p:+.4f}   d* = {d:+.4f}   gap = {p - d:.4f}")

### Verify numerically

In [ ]:
check("7.1 example: p* = d* = 0", np.isclose(pd1[0], 0, atol=1e-6) and np.isclose(pd1[1], 0, atol=1e-6))
check("non-convex f₀ with x ≤ 0.5: no gap (the global minimiser x ≈ −1.30 is feasible)", np.isclose(pd2[0], pd2[1], atol=1e-6))
check("non-convex f₀ with x ≥ 0.5: positive duality gap", pd3[0] - pd3[1] > 0.1)
check("weak duality d* ≤ p* in all three", all(d <= p + 1e-9 for p, d in [pd1, pd2, pd3]))

---
### Explain

**1. Read $p^\star$ and $d^\star$ directly on the picture.**

$p^\star$ is the height of the **lowest point of $\mathcal G$ in the half-plane $u\le0$** (feasible $x$ have $f_1(x)\le0$), the red star and red dashed line. $g(\lambda)$ is the $t$-intercept of the lowest line of slope $-\lambda$ that still touches $\mathcal G$ (all of $\mathcal G$ lies above it), and $d^\star$ is the **highest such intercept** over $\lambda\ge0$, the blue dotted line. For the 7.1 example the horizontal line ($\lambda = 1$) touches the parabola at $(0,0)$: $p^\star = d^\star = 0$. For $f_0 = x^4 - 3x^2 + x$ with $f_1 = x - 0.5$, both are $\approx-3.51$: the constraint $x\le0.5$ does not exclude the global minimiser $x\approx-1.30$, so $\lambda = 0$ already gives the optimal (horizontal) supporting line and there is **no gap**, despite the non-convexity. With the reversed constraint $x\ge0.5$ (third panel), $p^\star\approx-1.07$ (the right valley) while $d^\star$ is clearly lower: a **duality gap**.

**2. Why does a non-convex $\mathcal G$ produce a duality gap?**

A supporting line can only "see" the **convex hull** of $\mathcal G$ from below: $g(\lambda)$ depends on $\mathcal G$ only through its lower convex envelope. When the optimal feasible point (the lowest point with $u\le0$) lies in a "dent" of $\mathcal G$, a non-convex part above the convex hull, no line touches $\mathcal G$ there. Every line of slope $-\lambda\le0$ that passes under $\mathcal G$ is held down by the other valley (the infeasible global minimum region), so its intercept stays below $p^\star$. In the third panel, the feasible minimum $(u,t)\approx(-0.63, -1.07)$ sits above the segment joining the two valleys, and $d^\star$ is the height of that convex envelope at $u = 0$. In the second panel the feasible minimum is itself a point of the convex envelope, so the gap closes. For convex problems $\mathcal G$ (more precisely its "epigraph" $\mathcal G + \mathbb R^2_+$) is convex, and under Slater the supporting line reaches $p^\star$: no gap.

**3. Why is $\lambda$ restricted to be non-negative?**

The line $t + \lambda u = g$ must lie below $\mathcal G$ *and* give a lower bound on the feasible values, which have $u\le0$. With $\lambda\ge0$ and $u\le0$, $t\ge g - \lambda u\ge g$: every feasible point is at least $g$. With $\lambda<0$ the line slopes upwards to the left, and feasible points far to the left could be below its intercept, so it would no longer bound $p^\star$. Algebraically, $\lambda f_1(x)\le0$ on the feasible set (needed for $L\le f_0$) requires $\lambda\ge0$ for inequality constraints $f_1\le0$. (Equality-constraint multipliers $\mu$ are free, because $h(x) = 0$ there.)

---
## 7.4 Equivalent problems, different duals

**Task.** The problem $\min x^2$ s.t. $x\ge1$ ($p^\star = 1$, $x^\star = 1$) can be written with three different constraints describing the same feasible set $[1,\infty)$:

(a) $1 - x\le0$, (b) $-\log x\le0$ ($\mathcal D = \mathbb R_{++}$), (c) $(1-x)^3\le0$.

For each, plot $L(x,\lambda)$ for $\lambda = 0, 1, 2, 4$, compute $g(\lambda)$ (numerically on a grid and in closed form) and plot it together with $p^\star$.

### Prediction

- (a) $g = \lambda - \lambda^2/4$, max 1 at $\lambda = 2$. (b) $g = \frac\lambda2 - \frac\lambda2\log\frac\lambda2$, also max 1 at $\lambda = 2$. (c) For $\lambda>0$ the cubic $-\lambda x^3$ dominates as $x\to+\infty$: $g = -\infty$; $g(0) = 0$. So $d^\star = 0 < 1$: a gap.

In [ ]:
forms = {
    "(a) 1 − x ≤ 0": (lambda x: 1 - x, np.linspace(-20, 20, 400_001), lambda l: l - l**2 / 4),
    "(b) −log x ≤ 0": (lambda x: -np.log(x), np.linspace(1e-6, 20, 400_001),
                       lambda l: np.where(l > 0, l / 2 - l / 2 * np.log(np.where(l > 0, l / 2, 1)), 0.0)),
    "(c) (1 − x)³ ≤ 0": (lambda x: (1 - x) ** 3, np.linspace(-20, 20, 400_001), lambda l: np.where(l > 0, -np.inf, 0.0)),
}
lams = np.linspace(0, 6, 61)
fig, axes = plt.subplots(2, 3, figsize=(21, 9))
results = {}
for j, (name, (fc, xs, g_cf)) in enumerate(forms.items()):
    xp = xs[(xs > -1.5) & (xs < 4)]
    for l in [0, 1, 2, 4]:
        axes[0, j].plot(xp, xp**2 + l * fc(xp), label=f"λ = {l}")
    axes[0, j].axvline(1, color="k", ls=":"); axes[0, j].set_ylim(-6, 12); axes[0, j].legend(); axes[0, j].set_title(f"{name}: L(x, λ)")
    g_num = np.array([np.min(xs**2 + l * fc(xs)) for l in lams])
    results[name] = (g_num, g_cf(lams))
    axes[1, j].plot(lams, np.maximum(g_num, -10), "o", ms=4, label="g numerical (grid x ∈ D, clipped at −10)")
    axes[1, j].plot(lams, np.maximum(g_cf(lams), -10), "k", label="closed form")
    axes[1, j].axhline(1, color="r", ls="--", label="p* = 1"); axes[1, j].set_ylim(-11, 2); axes[1, j].legend(fontsize=8)
    axes[1, j].set_xlabel("λ"); axes[1, j].set_title(f"d* ≈ {g_num.max():.4f} at λ = {lams[g_num.argmax()]:.2f}")
plt.tight_layout(); plt.show()

### Verify numerically

In [ ]:
ga, gb, gc = results["(a) 1 − x ≤ 0"], results["(b) −log x ≤ 0"], results["(c) (1 − x)³ ≤ 0"]
check("(a) g(λ) = λ − λ²/4 (grid vs closed form)", np.allclose(ga[0], ga[1], atol=1e-6))
check("(b) g(λ) = λ/2 − (λ/2) log(λ/2)", np.allclose(gb[0], gb[1], atol=1e-6))
check("(a) and (b): λ* = 2 and d* = 1 = p*", np.isclose(lams[ga[0].argmax()], 2) and np.isclose(lams[gb[0].argmax()], 2)
      and np.isclose(ga[0].max(), 1) and np.isclose(gb[0].max(), 1))
check("(c): grid minimum keeps falling as the grid grows (→ −∞ for λ > 0)",
      np.min((np.linspace(-200, 200, 4001)) ** 2 + (1 - np.linspace(-200, 200, 4001)) ** 3) < gc[0][10] < 0)
check("(c): d* = g(0) = 0 < p* = 1 (duality gap 1)", np.isclose(gc[0][0], 0) and np.all(gc[0][1:] < -100))

---
### Explain

**1. Show that (a) gives $g(\lambda) = \lambda - \lambda^2/4$ and (b) gives $g(\lambda) = \frac\lambda2 - \frac\lambda2\log\frac\lambda2$. Are their maximisers $\lambda^\star$ the same?**

(a) $L = x^2 + \lambda(1-x)$, $\partial_xL = 2x - \lambda = 0\Rightarrow x = \lambda/2$, $g = \frac{\lambda^2}{4} + \lambda - \frac{\lambda^2}{2} = \lambda - \frac{\lambda^2}{4}$; $g' = 1 - \frac\lambda2 = 0\Rightarrow\lambda^\star = 2$, $d^\star = 1$.
(b) $L = x^2 - \lambda\log x$ on $x>0$, $\partial_xL = 2x - \lambda/x = 0\Rightarrow x^2 = \lambda/2$, $g = \frac\lambda2 - \lambda\log\sqrt{\lambda/2} = \frac\lambda2 - \frac\lambda2\log\frac\lambda2$; $g' = -\frac12\log\frac\lambda2 = 0\Rightarrow\lambda^\star = 2$, $d^\star = 1$.
Both match the grid computation. The dual *functions* are different (one is a parabola, the other is not), but **here the maximisers happen to coincide**, $\lambda^\star = 2$. That is a coincidence of this example: by KKT, $\lambda^\star = -f_0'(x^\star)/f_1'(x^\star) = 2/|f_1'(1)|$, and $f_1'(1) = -1$ in both (a) and (b). Rescaling a constraint, e.g. $2(1-x)\le0$, would rescale $\lambda^\star$ to 1 while describing the same set. So multipliers depend on how the constraint is written, not just on the feasible set.

**2. What is $g(\lambda)$ in (c), and why?**

$L = x^2 + \lambda(1-x)^3$. For any $\lambda>0$ the cubic term $-\lambda x^3$ dominates $x^2$ as $x\to+\infty$, so $L\to-\infty$: $g(\lambda) = -\infty$ for every $\lambda>0$ (the grid minimum gets more negative whenever the grid is enlarged). Only $g(0) = \inf x^2 = 0$ is finite. Hence $d^\star = 0 < p^\star = 1$: a **duality gap of 1**, for a problem whose optimal point is perfectly ordinary.

**3. Which of the three formulations is a convex problem in standard form [CO 10], and how does this explain the duality gaps you observe [CO 14.2.3]?**

(a) is convex (affine constraint), and Slater holds ($x = 2$ gives $1-2<0$), so strong duality holds: $d^\star = p^\star = 1$. (b) is also a convex problem in standard form: $-\log x$ is convex on $\mathcal D = \mathbb R_{++}$, with strict feasibility at $x = 2$, so again no gap. (c) is **not** in convex standard form: $(1-x)^3$ is not a convex function on $\mathbb R$ ($\frac{d^2}{dx^2} = 6(1-x) < 0$ for $x>1$, exactly on the feasible side). Its sublevel set happens to be convex ($[1,\infty)$), but the Lagrangian sees the *function*, not the set, and the non-convex constraint lets $L$ escape to $-\infty$. This is the lesson of [CO 14.2.3]: equivalent primal problems can have very different duals, and strong-duality guarantees need convex *functions* in standard form (plus a constraint qualification), not merely a convex feasible set.

---
## 7.5 Dual of a projection onto a halfspace

**Task.** Project a point $a\in\mathbb R^2$ onto the halfspace $\{x : c^\top x\le d\}$:

$$\min_x\ \|x-a\|_2^2\quad\text{s.t.}\quad c^\top x\le d, \qquad L(x,\lambda) = \|x-a\|_2^2 + \lambda(c^\top x - d).$$

Derive $x(\lambda) = \arg\min_xL$ and $g(\lambda) = -\frac{\lambda^2}{4}\|c\|_2^2 + \lambda(c^\top a - d)$. Plot the halfspace, $a$, the path $x(\lambda)$ for $\lambda\in[0,\lambda_{\max}]$, and, in a second panel, $g(\lambda)$ with $p^\star$. Move $a$ with sliders.

Derivation: $\nabla_xL = 2(x-a) + \lambda c = 0\Rightarrow x(\lambda) = a - \frac\lambda2c$; substituting, $g(\lambda) = \frac{\lambda^2}{4}\|c\|^2 + \lambda(c^\top a - \frac\lambda2\|c\|^2 - d) = -\frac{\lambda^2}{4}\|c\|^2 + \lambda(c^\top a - d)$.

### Prediction

- $x(\lambda)$ moves from $a$ straight along $-c$ (perpendicular to the boundary). It hits the boundary exactly at $\lambda^\star = 2(c^\top a - d)/\|c\|^2$, which is where $g$ peaks, at height $p^\star = \mathrm{dist}(a, H)^2$.
- If $a$ is already feasible, $g$ is decreasing for $\lambda\ge0$: $\lambda^\star = 0$, $p^\star = 0$.

In [ ]:
c_h, d_h = np.array([1.0, 2.0]), 2.0

def project_halfspace(a, c=c_h, d=d_h):
    lam_star = 2 * max(0.0, c @ a - d) / (c @ c)
    return a - lam_star / 2 * c, lam_star

def plot_projection(a1=2.5, a2=2.0):
    a = np.array([a1, a2]); x_star, lam_star = project_halfspace(a)
    p_star = np.sum((x_star - a) ** 2)
    lam_max = max(2 * lam_star, 1.0); lam = np.linspace(0, lam_max, 200)
    path = a[:, None] - lam / 2 * c_h[:, None]
    g = -lam**2 / 4 * (c_h @ c_h) + lam * (c_h @ a - d_h)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    G1, G2 = np.meshgrid(np.linspace(-3, 4, 300), np.linspace(-3, 4, 300))
    ax1.contourf(G1, G2, c_h[0] * G1 + c_h[1] * G2 <= d_h, levels=[0.5, 1.5], colors=["#cfe0f3"])
    ax1.plot(*path, "C1", lw=2, label="x(λ) = a − (λ/2)c, λ ∈ [0, λmax]")
    ax1.plot(*a, "ko", ms=9, label=f"a = {a}"); ax1.plot(*x_star, "r*", ms=15, label=f"projection x(λ*) = {x_star.round(3)}")
    ax1.set_aspect("equal"); ax1.set_xlim(-3, 4); ax1.set_ylim(-3, 4); ax1.legend(fontsize=8); ax1.set_title("halfspace cᵀx ≤ d (blue)")
    ax2.plot(lam, g, label="g(λ)"); ax2.axhline(p_star, color="r", ls="--", label=f"p* = {p_star:.3f}")
    ax2.plot(lam_star, -lam_star**2 / 4 * (c_h @ c_h) + lam_star * (c_h @ a - d_h), "r*", ms=15, label=f"λ* = {lam_star:.3f}")
    ax2.legend(); ax2.grid(alpha=0.3); ax2.set_xlabel("λ")
    plt.show()

@interact(a1=FloatSlider(min=-3, max=4, step=0.1, value=2.5), a2=FloatSlider(min=-3, max=4, step=0.1, value=2.0))
def explore_projection(a1, a2):
    plot_projection(a1, a2)

### Verify numerically

In [ ]:
rng = np.random.default_rng(0)
ok = True
for _ in range(200):
    a = rng.uniform(-3, 4, 2)
    x_star, lam_star = project_halfspace(a)
    num = minimize(lambda x: np.sum((x - a) ** 2), a, constraints=[{"type": "ineq", "fun": lambda x: d_h - c_h @ x}], method="SLSQP").x
    lam_grid = np.linspace(0, 10, 100_001)
    g = -lam_grid**2 / 4 * (c_h @ c_h) + lam_grid * (c_h @ a - d_h)
    ok &= np.allclose(x_star, num, atol=1e-5) and np.isclose(lam_grid[g.argmax()], lam_star, atol=1e-4) \
          and np.isclose(g.max(), np.sum((x_star - a) ** 2), atol=1e-6)
check("200 random a: x(λ*) = SLSQP projection, λ* = 2 max(0, cᵀa − d)/‖c‖² = argmax g, d* = p*", ok)

### Explore: a second halfspace $c_2^\top x\le d_2$ and $g(\lambda_1,\lambda_2)$

In [ ]:
c2, d2 = np.array([1.0, -1.0]), 0.5
Cm, dv = np.vstack([c_h, c2]), np.array([d_h, d2])

def g2(l, a):           # x(λ) = a − ½Cᵀλ,  g(λ) = −¼‖Cᵀλ‖² + λᵀ(Ca − d)
    return -0.25 * np.sum((Cm.T @ l) ** 2, axis=0) + l.T @ (Cm @ a - dv) if l.ndim == 1 else \
           -0.25 * np.sum((np.tensordot(Cm.T, l, 1)) ** 2, axis=0) + np.tensordot(Cm @ a - dv, l, 1)

def two_halfspaces(a1=2.5, a2=2.0):
    a = np.array([a1, a2])
    res = minimize(lambda l: -g2(l, a), x0=np.ones(2), bounds=[(0, None)] * 2)
    lam_s = res.x; x_s = a - 0.5 * Cm.T @ lam_s
    L1, L2 = np.meshgrid(np.linspace(0, 4, 120), np.linspace(0, 4, 120))
    Gv = g2(np.stack([L1, L2]), a)
    fig = plt.figure(figsize=(15, 6))
    ax = fig.add_subplot(1, 2, 1, projection="3d"); ax.plot_surface(L1, L2, Gv, cmap="viridis", lw=0, alpha=0.85)
    ax.scatter(*lam_s, -res.fun, color="r", s=80); ax.set_xlabel("λ₁"); ax.set_ylabel("λ₂"); ax.set_title(f"g(λ₁, λ₂), maximiser λ* = {lam_s.round(3)}")
    ax2 = fig.add_subplot(1, 2, 2)
    G1, G2 = np.meshgrid(np.linspace(-3, 4, 300), np.linspace(-3, 4, 300))
    ax2.contourf(G1, G2, (c_h[0] * G1 + c_h[1] * G2 <= d_h) & (c2[0] * G1 + c2[1] * G2 <= d2), levels=[0.5, 1.5], colors=["#cfe0f3"])
    ax2.plot(*a, "ko"); ax2.plot(*x_s, "r*", ms=15); ax2.plot(*np.column_stack([a, x_s]), "k:")
    ax2.set_aspect("equal"); ax2.set_title(f"projection onto the intersection: {x_s.round(3)};  p* = d* = {-res.fun:.3f}")
    plt.show()
    return a, lam_s, x_s, -res.fun

@interact(a1=FloatSlider(min=-3, max=4, step=0.1, value=2.5), a2=FloatSlider(min=-3, max=4, step=0.1, value=2.0))
def explore_two(a1, a2):
    two_halfspaces(a1, a2)

In [ ]:
a = np.array([2.5, 2.0])
xv = minimize(lambda x: np.sum((x - a) ** 2), a, constraints=[{"type": "ineq", "fun": lambda x: dv - Cm @ x}], method="SLSQP").x
res = minimize(lambda l: -g2(l, a), x0=np.ones(2), bounds=[(0, None)] * 2); lam_s = res.x
check("two halfspaces: x(λ*) = SLSQP projection and d* = p*", np.allclose(a - 0.5 * Cm.T @ lam_s, xv, atol=1e-4)
      and np.isclose(-res.fun, np.sum((xv - a) ** 2), atol=1e-6))
check("complementary slackness λᵢ*(cᵢᵀx* − dᵢ) = 0", np.allclose(lam_s * (Cm @ xv - dv), 0, atol=1e-4))

---
### Explain

**1. Show that $\lambda^\star = 2\max(0, c^\top a - d)/\|c\|_2^2$ and that $x(\lambda^\star)$ is the projection.**

$g$ is a concave parabola in $\lambda$ with unconstrained maximiser $g'(\lambda) = -\frac\lambda2\|c\|^2 + (c^\top a - d) = 0\Rightarrow\lambda = 2(c^\top a - d)/\|c\|^2$. With the constraint $\lambda\ge0$: if that value is negative, $g$ is decreasing on $[0,\infty)$ and the maximum is at $\lambda = 0$. Hence $\lambda^\star = 2\max(0, c^\top a - d)/\|c\|^2$. Then $x(\lambda^\star) = a - \frac{c^\top a - d}{\|c\|^2}c$ when $a$ is infeasible: it moves from $a$ along the normal $c$ by exactly the signed distance to the boundary, and $c^\top x(\lambda^\star) = d$. That is the orthogonal projection onto the hyperplane, which is the projection onto the halfspace (the closest point of a convex set to an outside point lies on its boundary, reached along the normal). $d^\star = g(\lambda^\star) = \frac{(c^\top a - d)^2}{\|c\|^2} = \mathrm{dist}(a, H)^2 = p^\star$. All checked on 200 random points.

**2. Why is $\lambda^\star = 0$ when $a$ is feasible?**

Then $a$ itself is the projection ($p^\star = 0$) and the constraint is not active at the solution. By complementary slackness $\lambda^\star(c^\top x^\star - d) = 0$ with $c^\top x^\star = c^\top a<d$ forces $\lambda^\star = 0$. In the dual picture, $c^\top a - d<0$ makes $g$ strictly decreasing for $\lambda\ge0$, so any positive multiplier only lowers the bound: no "price" needs to be paid for a constraint that is not binding. In the path plot, $x(\lambda)$ starting at a feasible $a$ moves *away* from the optimum.

**3. With two halfspaces, when is one of the multipliers zero at the maximiser of $g$?**

When that constraint is **inactive** at the projection, i.e. the projection onto the intersection lies strictly inside that halfspace (complementary slackness, checked numerically). Three regimes appear as you move $a$: (i) $a$ feasible: $\lambda^\star = (0,0)$; (ii) $a$ violates (essentially) one constraint and its projection onto that boundary satisfies the other: only that multiplier is positive, and the maximiser of $g$ sits on an edge of the quadrant $\lambda\ge0$; (iii) $a$ lies in the "corner region" (the normal cone of the vertex): the projection is the vertex where both boundaries meet, both multipliers are positive, and the maximiser is interior to the quadrant. Note that a multiplier can be zero even when $a$ violates that constraint, if projecting onto the other halfspace already fixes the violation.